# 02. Phase 2 — 학습: 경량 포트홀 탐지기

**CSE36501 Deep Learning (UNIST, Fall 2026)** · 정영찬 (20221355), 조수민 (20221371)

이 노트북 하나로 **환경 점검 → 데이터 확인 → 빠른 점검 → 본 학습 → 결과 모아보기**를 합니다.
모델 구조 설명은 `01_model_overview.ipynb`를 먼저 보세요.

**사용 규칙**
- 위에서부터 순서대로 실행합니다.
- 오래 걸리는 셀은 `RUN_... = False`가 기본이라, *Run All*을 눌러도 학습이 시작되지 않습니다. 직접 `True`로 바꿔야 합니다.
- 실제 학습 코드는 `src/train/train.py`에 있고, 노트북은 그것을 실행만 합니다 → 터미널에서도 똑같이 돌릴 수 있습니다.
- 이미 끝난 실험은 자동으로 건너뜁니다. 중간에 끊겨도 같은 셀을 다시 실행하면 이어서(남은 run부터) 진행합니다.
  단, *학습 도중* 끊긴 run 하나는 처음부터 다시 학습합니다.


## 1. 환경 점검 — 라이브러리 호환성 문제 해결

### 무엇이 문제였나
이전 `requirements.txt`에는 아래 문제가 있었고, 지금은 모두 고쳤습니다.

| 원인 | 증상 | 해결 |
|---|---|---|
| `torch==2.11.0+cu128` 처럼 **CUDA 빌드 태그까지 고정** | Mac·CPU 환경은 설치 실패, 드라이버가 오래된 PC는 GPU를 못 씀 | torch는 **하드웨어에 맞게 따로 먼저 설치**, requirements에는 태그 없이 버전만 |
| `thop` 와 `ultralytics-thop` 가 **같은 `thop` 폴더**에 설치됨 | 서로 덮어써서 GFLOPs가 조용히 0이 됨 | `thop` 삭제 (ultralytics가 `ultralytics-thop`을 알아서 설치) |
| `opencv-python` 과 `opencv-python-headless` 가 **같은 `cv2` 폴더**에 설치됨 | 버전이 다르면 `import cv2` 오류 | 두 패키지를 **같은 버전으로 고정** |
| Python 버전·기존 환경(Anaconda 등)과 충돌 | 설치 중 빌드 실패, 버전 충돌 | **새 가상환경 + Python 3.11~3.13** |
| **알파 버전 Python(3.16.0a1)으로 만든 가상환경** — VS Code가 만든 `deep_learning` 환경이 이 경우 | numpy·torch 설치 실패 → `No module named 'numpy'` | Python **3.11**로 가상환경을 다시 만들기 |

> **우리 PC에서 확인된 실제 원인**: 프로젝트 안의 `deep_learning` 가상환경은 설정 파일엔 3.11로 적혀 있지만
> 실제로는 **Python 3.16.0a1(알파)** 로 실행됩니다. 알파 버전용 numpy·torch 설치 파일이 없어서 설치가 실패했습니다.
> 이미 모든 패키지가 설치·검증된 **`.venv`(Python 3.11)** 를 쓰거나(VS Code 오른쪽 위 커널 선택 →
> `Python 3.11 (.venv pothole)`), `deep_learning`을 3.11로 다시 만드세요.

확인: Windows / Linux / Apple Silicon Mac × Python 3.11~3.14 모두 미리 빌드된 패키지(wheel)만으로 설치되는 것을 확인했습니다.
(단, Intel Mac은 PyTorch가 더 이상 지원하지 않습니다.)

### 올바른 설치 순서 (터미널, 프로젝트 루트에서)
```bash
# 0) 새 가상환경 (uv 사용 시)
uv venv --python 3.11 .venv
#    또는 python -m venv .venv  (Python 3.11~3.13)
# 1) PyTorch — 하드웨어에 맞는 것 하나만
#    NVIDIA, 드라이버 570 이상 (nvidia-smi 오른쪽 위 "Driver Version" 확인)
pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
#    NVIDIA, 드라이버가 더 오래됨
pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu126
#    GPU 없음 / Mac
pip install torch==2.11.0 torchvision==0.26.0
# 2) 나머지
pip install -r requirements.txt
# 3) 이 노트북의 커널 등록
python -m ipykernel install --user --name pothole-venv --display-name "Python 3.11 (.venv pothole)"
```
`uv`로 만든 가상환경에는 `pip`이 없으므로 `pip install ...` 대신 `uv pip install --python .venv/Scripts/python.exe ...` 를 씁니다
(Mac/Linux는 `.venv/bin/python`).

아래 셀이 현재 환경을 진단하고, 문제가 있으면 **고치는 명령어**를 알려 줍니다.


In [ ]:
import importlib.metadata as md_
import platform, re, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

def ver(name):
    try:
        return md_.version(name)
    except md_.PackageNotFoundError:
        return None

checks = []  # (상태, 항목, 내용, 해결 방법)
def add(ok, item, detail, fix=""):
    checks.append(("✅" if ok is True else "⚠️" if ok is None else "❌", item, detail, fix))

py = sys.version_info
if py.releaselevel != "final":
    add(False, "Python", f"{platform.python_version()} (정식 출시 전 {py.releaselevel} 버전)",
        "알파/베타 Python에는 numpy·torch 설치 파일이 없습니다 → Python 3.11로 가상환경을 다시 만드세요: "
        "uv venv --python 3.11 .venv")
else:
    add(py[:2] in [(3, 11), (3, 12), (3, 13)], "Python", platform.python_version(),
        "" if py[:2] in [(3, 11), (3, 12), (3, 13)] else "Python 3.11~3.13으로 새 가상환경을 만드세요")
cfg_file = Path(sys.prefix) / "pyvenv.cfg"
if cfg_file.exists():
    m = re.search(r"^version(?:_info)?\s*=\s*(\d+\.\d+)", cfg_file.read_text(encoding="utf-8"), re.M)
    if m and m.group(1) != f"{py[0]}.{py[1]}":
        add(False, "가상환경 일치", f"pyvenv.cfg는 Python {m.group(1)}, 실제 실행은 {py[0]}.{py[1]}",
            "망가진 가상환경입니다 → 지우고 다시 만드세요 (uv venv --python 3.11 <폴더> --clear)")
add(sys.prefix != sys.base_prefix or None, "가상환경", sys.prefix,
    "" if sys.prefix != sys.base_prefix else "가상환경이 아닙니다. 기존 패키지와 충돌할 수 있으니 .venv를 쓰세요")

try:
    import torch
    cuda_build = torch.version.cuda
    add(True, "PyTorch", f"{torch.__version__} (CUDA 빌드: {cuda_build or '없음/CPU'})")
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True)
    has_nvidia = smi.returncode == 0
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        vram = p.total_memory / 2**30
        add(True, "GPU", f"{p.name}, VRAM {vram:.1f} GB")
        if vram < 10:
            add(None, "VRAM", f"{vram:.1f} GB", "8GB급: 기본 batch=-1(자동)로 시작, 메모리 부족 시 configs/train.yaml에서 batch: 16 → 8")
    elif has_nvidia:
        drv = smi.stdout.strip().split(",")[1].strip()
        if not cuda_build:
            fix = "CPU용 torch가 설치됨 → pip uninstall -y torch torchvision 후 위 1)의 NVIDIA 명령으로 재설치"
        else:
            fix = f"드라이버 {drv}가 CUDA {cuda_build}보다 오래됨 → 드라이버 업데이트 또는 cu126 버전 torch 설치"
        add(False, "GPU", f"NVIDIA GPU는 있으나 PyTorch가 못 씀 ({smi.stdout.strip()})", fix)
    else:
        add(None, "GPU", "NVIDIA GPU 없음 → CPU로만 학습 가능 (매우 느림)")
except ImportError:
    add(False, "PyTorch", "설치 안 됨", "위 설치 순서 1)을 먼저 실행")

ocv, ocvh = ver("opencv-python"), ver("opencv-python-headless")
add(ocv == ocvh or None if ocv and ocvh else True, "OpenCV",
    f"opencv-python {ocv}, opencv-python-headless {ocvh}",
    "" if not (ocv and ocvh) or ocv == ocvh else
    f"pip install --force-reinstall opencv-python=={ocvh} opencv-python-headless=={ocvh}")
try:
    import cv2
    add(True, "import cv2", cv2.__version__)
except Exception as e:
    add(False, "import cv2", repr(e)[:80], "pip install --force-reinstall opencv-python opencv-python-headless (같은 버전)")

old_thop = ver("thop")
add(old_thop is None, "thop 충돌", f"thop {old_thop}, ultralytics-thop {ver('ultralytics-thop')}",
    "" if old_thop is None else "pip uninstall -y thop ultralytics-thop && pip install ultralytics-thop")

# requirements.txt에 고정한 버전과 실제 설치 버전 비교
mismatch = []
for line in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    m = re.match(r"^([A-Za-z0-9_.\-]+)==([^\s#]+)", line.strip())
    if m:
        name, want = m.groups()
        have = ver(name)
        if have is None or have.split("+")[0] != want:
            mismatch.append(f"{name}: 필요 {want}, 설치 {have}")
add(not mismatch, "requirements 버전", "모두 일치" if not mismatch else "; ".join(mismatch[:6]),
    "" if not mismatch else "pip install -r requirements.txt")

try:
    from ultralytics.utils.torch_utils import get_flops
    from ultralytics.nn.tasks import DetectionModel
    g = get_flops(DetectionModel("yolo26n.yaml", nc=1, verbose=False), 640)
    add(g > 0, "Ultralytics GFLOPs 계산", f"yolo26n = {g:.2f}", "" if g > 0 else "thop 충돌 해결 필요 (위 항목)")
except Exception as e:
    add(False, "Ultralytics", repr(e)[:80], "pip install -r requirements.txt")

try:  # 망가진 환경에서도 결과를 보여 주도록 pandas 없이도 동작
    import pandas as pd
    pd.set_option("display.max_colwidth", 120)
    display(pd.DataFrame(checks, columns=["상태", "항목", "내용", "해결 방법"]))
except ImportError:
    for c in checks:
        print(" | ".join(c))
print("❌가 있으면 '해결 방법'의 명령을 터미널에서 실행한 뒤, 커널을 재시작하고 이 셀을 다시 실행하세요.")
print("그래도 안 되면 오류 메시지 전체를 복사해서 공유해 주세요.")


## 2. Phase 2 실험 설계 — 효율화 사다리

기준점은 **`yolo26n` + 전체 프레임 + 640 입력**입니다. 각 실험은 기준점에서 **딱 하나만** 바꿉니다
→ 어떤 수단이 정확도를 얼마나 잃고 연산을 얼마나 줄이는지 분리할 수 있습니다 (**RQ2**).

| 실험 | 바꾸는 것 | 답하는 질문 |
|---|---|---|
| **E1** | 모델 계열·크기: YOLOv8n / v8s / YOLO26n / 26s | RQ1: 얼마나 작아도 되나 |
| **E2** | 입력 해상도 640 → 480 → 320 | RQ2: 해상도 축소의 대가 (특히 작은 포트홀) |
| **E3** | 하늘 잘라내기 (아래 60%만) | RQ2: 포트홀 크기는 유지하며 연산 약 40% 절감 |
| **E4** | P2 출력 가지 추가 (전체 / 잘라낸 입력) | RQ2: 작은 포트홀 회복 vs 연산 증가 |
| **E5** | 지식 증류 (학생 yolo26n ← 교사 yolo26s) | RQ2: 실행 비용 0으로 정확도 회복 (E1 끝난 뒤) |
| E0 | RT-DETR-L (선택) | 정확도 상한 참고. 무겁고 느림 |

- 모든 실험은 시드 **42, 43, 44** 세 번 (평균 ± 표준편차). 시간이 부족하면 시드 42만 먼저 돌리세요.
- FP16/INT8 변환과 실제 속도(후처리 포함), 테스트셋 A~D 평가는 **Phase 3**에서 합니다.
- 설정 파일: `configs/experiments.yaml`(실험), `configs/train.yaml`(공통 하이퍼파라미터: epochs 100, patience 20, cos_lr 등)


In [ ]:
import yaml
exp = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text(encoding="utf-8"))
tcfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text(encoding="utf-8"))
rows = [{"실험": k, "설명": v["desc"], "모델": ", ".join(v["models"]),
         "데이터": v["data"] if isinstance(v["data"], str) else ", ".join(v["data"]),
         "입력": ", ".join(map(str, v["imgsz"])), "시드": ", ".join(map(str, v["seeds"])),
         "교사": v.get("teacher", "")} for k, v in exp["experiments"].items()]
display(pd.DataFrame(rows).set_index("실험"))
print({k: tcfg[k] for k in ["epochs", "patience", "batch", "optimizer", "cos_lr", "cache"]})


## 3. 데이터 확인

Phase 1 데이터(`make data`)와 잘라낸 데이터셋(`python -m src.data.road_crop`)이 있어야 합니다.
데이터는 저장소에 없으므로, 처음이라면 README의 **Data** 절대로 받은 뒤 아래 명령을 실행합니다.
```bash
make data                         # 또는 python -m src.data.inspect ... (Makefile 참고)
python -m src.data.road_crop      # data/processed/rdd_crop60 생성 (약 1~2분)
```


In [ ]:
from src.data.synth_rain import read_list

need = {"rdd.yaml": ROOT / "data/processed/rdd.yaml",
        "rdd_crop60.yaml": ROOT / "data/processed/rdd_crop60.yaml"}
ok = {k: p.exists() for k, p in need.items()}
print({k: ("있음" if v else "없음") for k, v in ok.items()})
if all(ok.values()):
    sp = ROOT / "data/processed/splits"
    counts = {}
    for name in ["rdd_train", "rdd_val", "rdd_test", "rdd_crop60_train", "rdd_crop60_val", "rdd_crop60_test"]:
        imgs = read_list(sp / f"{name}.txt")
        boxes = sum(len((p.parent.parent / "labels" / f"{p.stem}.txt").read_text().split("\n")) - 1 for p in imgs)
        counts[name] = {"이미지": len(imgs), "포트홀 박스": boxes}
    display(pd.DataFrame(counts).T)
else:
    print("데이터가 없습니다. 위 명령으로 먼저 만들어 주세요.")


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from matplotlib import font_manager
for name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = name; break
plt.rcParams["axes.unicode_minus"] = False

def show(ax, img_path, title):
    im = cv2.cvtColor(cv2.imdecode(np.fromfile(img_path, np.uint8), 1), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    ax.imshow(im)
    for r in (img_path.parent.parent / "labels" / f"{img_path.stem}.txt").read_text().splitlines():
        _, cx, cy, w, h = map(float, r.split())
        ax.add_patch(mpatches.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, ec="lime", lw=2))
    ax.set_title(f"{title}  ({W}×{H})"); ax.axis("off")

if all(ok.values()):
    stem = "India_005660"
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1]})
    show(axes[0], ROOT / "data/processed/rdd/images" / f"{stem}.jpg", "전체 프레임 (E1, E2)")
    show(axes[1], ROOT / "data/processed/rdd_crop60/images" / f"{stem}.jpg", "아래 60% (E3, E4)")
    plt.tight_layout(); plt.show()


## 4. 사전학습 가중치가 얼마나 옮겨지나

COCO로 사전학습된 가중치(`yolo26n.pt` 등)를 **이름과 모양이 같은 층**에만 불러옵니다.
1클래스 출력층과 새로 추가한 P2 가지는 모양이 달라 **무작위로 시작**합니다 (01 노트북 4절).

> 이 셀은 Ultralytics 공식 저장소에서 사전학습 가중치를 **다운로드**합니다
> (`yolo26n.pt` 약 5MB). 학습할 때도 어차피 받는 파일입니다.


In [ ]:
RUN_WEIGHT_CHECK = True   # 인터넷이 없으면 False

if RUN_WEIGHT_CHECK:
    import os
    (ROOT / "outputs/weights").mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT / "outputs/weights")  # 가중치 파일이 노트북 폴더에 흩어지지 않도록
    from ultralytics import settings
    settings.update({"sync": False})  # Ultralytics로 사용 통계를 보내지 않음
    from src.train.train import transferred_items
    rows = []
    for cfg, w in [("yolo26n.yaml", "yolo26n.pt"), ("yolo26n-p2.yaml", "yolo26n.pt")]:
        loaded, total = transferred_items(cfg, w)
        rows.append({"모델": cfg, "사전학습 가중치": w, "불러온 텐서": loaded, "전체 텐서": total,
                     "무작위 시작": total - loaded})
    os.chdir(ROOT)
    display(pd.DataFrame(rows))


## 5. 빠른 점검 (5% 데이터, 3 epoch)

본 학습 전에 **파이프라인이 끝까지 도는지** 확인합니다. 정확도는 의미 없습니다.
결과는 `outputs/runs_quick/`에 따로 저장되어 본 학습과 섞이지 않습니다.
RTX 3090 기준 약 1~2분, 8GB GPU는 조금 더 걸립니다.

학습은 노트북 커널이 아니라 **별도 프로세스**로 실행합니다 (Windows에서 멀티프로세싱 문제 회피, 메모리 정리).


In [ ]:
import os

def run(args):
    # src.train.train을 별도 프로세스로 실행하고 로그를 그대로 보여 줌
    cmd = [sys.executable, "-m", "src.train.train", *args]
    print("$", " ".join(cmd[1:]), flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", env=env)
    last = ""
    for chunk in proc.stdout:
        for part in chunk.split("\r"):          # 진행 막대(\r)는 마지막 상태만 출력
            if part.endswith("\n"):
                print(part, end="", flush=True)
            else:
                last = part
    if last.strip():
        print(last)
    rc = proc.wait()
    print(f"\n[종료 코드 {rc}]", "✅" if rc == 0 else "❌ 위 로그의 마지막 오류를 확인하세요")
    return rc

RUN_QUICK = False   # True로 바꾸고 실행

if RUN_QUICK:
    _ = run(["--exp", "E1", "--models", "yolo26n", "--seeds", "42", "--quick", "--yes"])


## 6. 전체 계획과 예상 시간

학습을 시작하지 않고 계획만 보여 줍니다 (`--list`). 예상 시간은 **이 컴퓨터에서 끝난 run의 실제 속도**로 계산되고,
아직 끝난 run이 없으면 RTX 3090 기준 대략값을 씁니다. RTX 3070은 대략 **2배** 걸린다고 보면 됩니다.
early stopping(20 epoch 동안 개선이 없으면 중단) 때문에 실제로는 보통 더 짧습니다.


In [ ]:
_ = run(["--exp", "E1", "E2", "E3", "E4", "E5", "--list"])


## 7. 본 학습

실험별로 `True`로 바꾸고 실행합니다. 권장 순서: **E1 → E3 → E4 → E2 → E5** (E5는 E1의 yolo26s가 교사).

- 시드 하나만 먼저: `SEEDS = ["42"]`. 세 시드 전부: `SEEDS = []`.
- 몇 시간짜리라 **터미널에서 돌리는 것도 좋습니다** (노트북 창을 닫아도 계속됨):
  ```bash
  python -m src.train.train --exp E1 --seeds 42 --yes
  ```
- 결과: `outputs/runs/{run_id}/weights/best.pt`, `run_meta.json`(요약), `results.csv`(epoch별 기록), `results.png`
- 메모리 부족(`CUDA out of memory`)이 나면 `configs/train.yaml`의 `batch: -1`을 `16` → `8`로 낮추세요.


In [ ]:
RUN = {"E1": False, "E3": False, "E4": False, "E2": False, "E5": False, "E0": False}
SEEDS = ["42"]          # [] 이면 configs/experiments.yaml의 시드 전부

for e, go in RUN.items():
    if go:
        rc = run(["--exp", e, *(["--seeds", *SEEDS] if SEEDS else []), "--yes"])
        if rc != 0:
            break


## 8. 결과 모아보기 (검증셋 기준)

각 run의 `run_meta.json`을 모아 **설정별 평균 ± 표준편차**(시드 기준)로 정리하고, 연산량 대비 정확도를 그립니다.
여기 숫자는 **검증셋(RDD val)** 기준입니다. 테스트셋 A~D 최종 평가와 속도 측정은 Phase 3에서 합니다.


In [ ]:
import json
metas = [json.loads(p.read_text(encoding="utf-8")) for p in (ROOT / "outputs/runs").glob("*/run_meta.json")]
if not metas:
    print("아직 끝난 run이 없습니다.")
else:
    df = pd.DataFrame(metas)
    df["설정"] = df["run_id"].str.replace(r"_s\d+$", "", regex=True)
    agg = df.groupby("설정").agg(실험=("exp", "first"), 시드수=("seed", "count"),
                                 params_M=("params_M", "first"), GFLOPs=("gflops", "first"),
                                 mAP50=("val_mAP50", "mean"), mAP50_std=("val_mAP50", "std"),
                                 mAP50_95=("val_mAP50_95", "mean"), mAP50_95_std=("val_mAP50_95", "std"),
                                 recall=("val_recall", "mean"), epochs=("epochs_done", "mean"),
                                 시간_h=("wall_hours", "mean")).sort_values("GFLOPs")
    display(agg.round(3))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for name, r in agg.iterrows():
        ax.errorbar(r["GFLOPs"], r["mAP50_95"], yerr=0 if pd.isna(r["mAP50_95_std"]) else r["mAP50_95_std"],
                    fmt="o", capsize=3, ms=4 + 2 * r["params_M"] ** 0.5)
        ax.annotate(name, (r["GFLOPs"], r["mAP50_95"]), xytext=(6, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel("GFLOPs (학습 입력 크기 기준, 작을수록 가벼움)"); ax.set_ylabel("검증 mAP50-95")
    ax.set_title("정확도 vs 연산량 (검증셋, 시드 평균 ± 표준편차)"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print("참고: 잘라낸 입력(crop60)의 실제 추론 GFLOPs는 위 값의 약 60%입니다 (Phase 3에서 직사각형 입력으로 측정).")


## 9. 8GB GPU(RTX 3070 등) 팁

| 상황 | 해결 |
|---|---|
| `CUDA out of memory` | `configs/train.yaml`의 `batch`를 `16` → `8`로. RT-DETR(E0)은 `4` |
| 학습이 너무 느림 | `--seeds 42`로 시드 하나만 먼저. E2(320/480)·E3는 빠름, yolov8s·yolo26s·E0가 느림 |
| RAM 부족 | `cache: ram` → `cache: false` (64GB면 문제없음) |
| Windows에서 데이터 로더 멈춤 | `workers: 8` → `4` 또는 `0` |
| 인터넷 없음 | 사전학습 가중치(`yolo26n.pt` 등)를 미리 `outputs/`에 넣거나, 파이프라인 점검만 `--offline`(무작위 시작) |

**용어**: epoch = 학습 데이터를 한 바퀴 / patience = 이만큼 개선이 없으면 조기 종료 /
batch = 한 번에 GPU에 올리는 이미지 수 / AutoBatch(`-1`) = GPU 메모리의 약 60%를 쓰도록 자동 결정
